# Which modality predicts PD best? Compare performance of modality-specific and combined models

In [ ]:
import numpy as np
import pandas as pd

import seaborn as sns
import pylab as plt

from scipy import stats
from statannot import add_stat_annotation
from ast import literal_eval
import re
#import pingouin as pg

from sklearn import metrics

from importlib import reload
import sys
import os
sys.path.insert(1,'../')
import plots

In [ ]:
reload(plots)
image_path = plots.set_dir()
colors = plots.set_colors()
data_path = '/scratch/c.c21013066/data/ukbiobank/analyses/acc_models/noOsteo'

In [ ]:
# chance_level performance
n_prodromal = 113
n_diagnosed = 153
n_unaff = 24987
n_pop = 33009
print('prod chance level matched: ', n_prodromal/(n_prodromal+n_prodromal))
print('diag chance level matched: ', n_diagnosed/(n_diagnosed+n_diagnosed))
print('prod chance level unmatched: ', n_prodromal/(n_prodromal+n_unaff))
print('diag chance level unmatched: ', n_diagnosed/(n_diagnosed+n_unaff))
print('prod chance level pop: ', n_prodromal/(n_prodromal+n_pop))
print('diag chance level pop: ', n_diagnosed/(n_diagnosed+n_pop))

In [ ]:
models = ['diag_PDHC','diag_ProdHC','diag_PDProdHC','diag_PDallHC','diag_ProdallHC','diag_PDProdallHC','diag_PDPopulationNoProd','diag_ProdPopulationNoPD',
          'diag_PDProdPopulation']
models0 = ['diag_PDHC','diag_ProdHC','diag_PDProdHC','diag_PDHC','diag_ProdHC','diag_PDProdHC','diag_PDPopulationNoProd','diag_ProdPopulationNoPD',
          'diag_PDProdPopulation']
kinds = ['HC','HC','HC','allHC','allHC','allHC','pop','pop','pop']
features = ['intercept','covariates','PRS','genetics+family','lifestyle_nofam','blood','prodromalsigns_beforePD','acc','all_acc_features',
            'genetics+family+all_acc_features','lifestyle+all_acc_features','blood+all_acc_features','prodromalsigns_beforePD+all_acc_features',
            'all_acc_features+blood+lifestyle+genetics+prodromalsigns_beforePD']
stacked_features = ['genetics+family+all_acc_features','lifestyle+all_acc_features','blood+all_acc_features','prodromalsigns_beforePD+all_acc_features',
            'all_acc_features+blood+lifestyle+genetics+prodromalsigns_beforePD']
stacked_features_names = [f'{n}_stacked' for n in stacked_features]

performances = pd.DataFrame(index=pd.MultiIndex.from_product([models,np.hstack([features,stacked_features_names])],names=['model','features']),
                            columns=np.arange(5))
performances_pr = pd.DataFrame(index=pd.MultiIndex.from_product([models,np.hstack([features,stacked_features_names])],names=['model','features']),
                               columns=np.arange(5))
curves = pd.DataFrame(index=pd.MultiIndex.from_product([models,np.hstack([features,stacked_features_names])],names=['model','features']),
                               columns=pd.MultiIndex.from_product([['tpr','fpr','precision','recall'],np.arange(5)],names=['metric','cv']))

for kind,modelname,model in zip(kinds,models,models0):
    for i,feature in enumerate(np.hstack([stacked_features_names,features])):
        print(modelname,feature)
        if 'stacked' in feature:
            name = modelname.replace('diag_','')
            fname = stacked_features[i]
            stacked = pd.read_csv(f'{data_path}/{fname}/stacked_perf.csv',index_col=[0,1],header=[0,1])
            stacked = stacked['test']
            performances.loc[(modelname,feature),:] = stacked.loc[name,'AUROC'].dropna().values
            performances_pr.loc[(modelname,feature),:] = stacked.loc[name,'AUPRC'].dropna().values
            curves.loc[(modelname,feature),('fpr',slice(None))] = stacked.loc[name,'fpr'].dropna().values
            curves.loc[(modelname,feature),('tpr',slice(None))] = stacked.loc[name,'tpr'].dropna().values
            curves.loc[(modelname,feature),('precision',slice(None))] = stacked.loc[name,'precision'].dropna().values
            curves.loc[(modelname,feature),('recall',slice(None))] = stacked.loc[name,'recall'].dropna().values
        else:
            if kind == 'HC':
                path = f'{data_path}/{feature}/HCmodels_perf_stacked_matched.csv'
                path2 = f'{data_path}/{feature}/HCmodels_curves_stacked_matched.csv'
            elif kind == 'allHC':
                path = f'{data_path}/{feature}/HCmodels_perf_stacked_allHC.csv'
                path2 = f'{data_path}/{feature}/HCmodels_curves_stacked_allHC.csv'
            elif kind == 'pop':
                path = f'{data_path}/{feature}/popmodelsNoPD_perf_stacked_allHC.csv'
                path2 = f'{data_path}/{feature}/popmodels_curves_stacked_allHC.csv'
            perf = pd.read_csv(path,index_col=[0,1],header=[0,1])
            perf = perf['test']
            perf_a = perf.loc[model,'ROCAUC'].dropna()
            performances.loc[(modelname,feature),:] = perf_a.values
            perf_p = perf.loc[model,'precision recall AUC'].dropna()
            performances_pr.loc[(modelname,feature),:] = perf_p.values
            curve =  pd.read_csv(path2,index_col=[0,1],header=[0,1])
            for cv in np.arange(5):
                curves.loc[(modelname,feature),('fpr',cv)] = curve.loc[(model,cv),'fpr'].dropna().values
                curves.loc[(modelname,feature),('tpr',cv)] = curve.loc[(model,cv),'tpr'].dropna().values
                curves.loc[(modelname,feature),('precision',cv)] = curve.loc[(model,cv),'precision'].dropna().values
                curves.loc[(modelname,feature),('recall',cv)] = curve.loc[(model,cv),'recall'].dropna().values
performances = performances.stack().reset_index()
performances_pr = performances_pr.stack().reset_index()
performances = performances.sort_values(0)
performances_pr = performances_pr.sort_values(0)
performances[0] = performances[0].astype(float)
performances_pr[0] = performances_pr[0].astype(float)

In [ ]:
pf = pd.DataFrame(columns=pd.MultiIndex.from_product([['AUROC','AUPRC'],['mean','std']],names=['metric','statistic']),
                  index=pd.MultiIndex.from_product([models,np.hstack([stacked_features_names,features])],names=['model','feature']))
for model in models:
    for feature in np.hstack([stacked_features_names,features]):
        pf.loc[(model,feature),('AUROC','mean')] = performances.loc[np.logical_and(performances['model']==model,
                                                                                   performances['features']==feature),
                                                                    0].mean()
        pf.loc[(model,feature),('AUROC','std')] = performances.loc[np.logical_and(performances['model']==model,
                                                                                   performances['features']==feature),
                                                                   0].std()#/np.sqrt(5)
        pf.loc[(model,feature),('AUPRC','mean')] = performances_pr.loc[np.logical_and(performances_pr['model']==model,
                                                                                   performances_pr['features']==feature)
                                                                       ,0].mean()
        pf.loc[(model,feature),('AUPRC','std')] = performances_pr.loc[np.logical_and(performances_pr['model']==model,
                                                                                   performances_pr['features']==feature)
                                                                      ,0].std()#/np.sqrt(5)

In [ ]:
# supplemental table
pf.to_csv('/scratch/c.c21013066/data/ukbiobank/analyses/acc_models/noOsteo/allmodels_perfNoPD_stacked.csv')

In [ ]:
pf = pd.read_csv('/scratch/c.c21013066/data/ukbiobank/analyses/acc_models/allmodels_perfNoPD_stacked.csv',header=[0,1],index_col=[0,1])
pvalues = pd.read_csv('/scratch/c.c21013066/data/ukbiobank/analyses/acc_models/pvaluesAUPRC_new.csv',header=[0,1],index_col=[0,1])

In [ ]:
pf.loc[('diag_ProdPopulationNoPD',slice(None)),'AUPRC']

In [ ]:
statistics.loc[(slice(None),'all_acc_features'),('diag_ProdPopulationNoPD')]

In [ ]:
#features = np.hstack([stacked_features_names,features])
statistics = pd.DataFrame(columns=pd.MultiIndex.from_product([models,['T','dof','alternative','p-value','CI95%',
                                                               'cohen-d','BF10','power']],names=['model','statistic']),
                       index=pd.MultiIndex.from_product([np.hstack([stacked_features_names,features]),np.hstack([stacked_features_names,features])],names=['modality1','modality2']))
pvalues = pd.DataFrame(columns=pd.MultiIndex.from_product([models,['T','dof','alternative','p-value','CI95%',
                                                               'cohen-d','BF10','power']],names=['model','statistic']),
                       index=pd.MultiIndex.from_product([np.hstack([stacked_features_names,features]),np.hstack([stacked_features_names,features])],names=['modality1','modality2']))

for i,f1 in enumerate(np.hstack([stacked_features_names,features])):
    for j,f2 in enumerate(np.hstack([stacked_features_names,features])):
            for model in models:
                statistics.loc[(f1,f2),(model,slice(None))] = pg.ttest(performances_pr.loc[np.logical_and(performances_pr['model']==model,
                                                                       performances_pr['features']==f1),0],
                                   performances_pr.loc[np.logical_and(performances_pr['model']==model,
                                                                      performances_pr['features']==f2),0],correction=False).values
                t,p = stats.ttest_ind(performances_pr.loc[np.logical_and(performances_pr['model']==model,
                                                                         performances_pr['features']==f1),0],
                                     performances_pr.loc[np.logical_and(performances_pr['model']==model,
                                                                        performances_pr['features']==f2),0])
                pvalues.loc[(f1,f2),(model,'p-value')] = p
                pvalues.loc[(f1,f2),(model,'T')] = t
pvalues = pvalues.dropna(axis='rows',how='all')
statistics = statistics.dropna(axis='rows',how='all')
# get pvalues that are significant
keep = pvalues.loc[:,(slice(None),'p-value')]<(0.05/pvalues.shape[0])
pvalues_sig = pvalues[keep]
# extract boxpairs from that
box_pairs = []
for model in models:
    sig = pvalues_sig[model].dropna(axis='rows',how='all',subset=['p-value'])
    for key,s in sig.iterrows():
        box_pairs.append(((model,key[0]),(model,key[1])))

In [ ]:
modalities = ['genetics+family','blood','lifestyle_nofam','prodromalsigns_beforePD','all_acc_features']
modalities2 = ['genetics+family+all_acc_features','blood+all_acc_features','lifestyle+all_acc_features','prodromalsigns_beforePD+all_acc_features',
              'all_acc_features+blood+lifestyle+genetics+prodromalsigns_beforePD']
modalities3 = [f'{n}_stacked' for n in modalities2]
comp = []
for m,m2,m3 in zip(modalities,modalities2,modalities3):
    comp.append(('intercept',m))
    comp.append((m,m2))
    comp.append((m,m3))

In [ ]:
# supplemental table
pvalues.to_csv('/scratch/c.c21013066/data/ukbiobank/analyses/acc_models/noOsteo/pvaluesAUPRC.csv')

In [ ]:
statistics.to_csv('/scratch/c.c21013066/data/ukbiobank/analyses/acc_models/noOsteo/pvaluesAUPRC_new.csv')

In [ ]:
order = ['intercept','genetics+family','lifestyle_nofam','blood','prodromalsigns_beforePD','all_acc_features',
         'all_acc_features+blood+lifestyle+genetics+prodromalsigns_beforePD']#,'all_acc_features+blood+lifestyle+genetics+prodromalsigns_beforePD_stacked']
labels = [['Diagnosed','Prodromal','Diagnosed &\nProdromal'],
                         ['Diagnosed','Prodromal','Diagnosed &\nProdromal'],['Diagnosed','Prodromal','Diagnosed &\nProdromal']]
titles = ['control = matched unaffected','control = unaffected', 'control = general population']
fig,axes = plt.subplots(nrows=1,ncols=3,figsize=(20,4))
plots.plot_context()
sns.set_style('whitegrid')

subplotlabelsA = ['A','B','C']

for col,control,label,title in zip(np.arange(3),['[d|D]HC','allHC','Population'],labels,titles):
    r = re.compile(f'.*{control}.*')
    models_ = list(filter(r.match, models))
    label = label[:-1]
    performances_pr_ = performances_pr[performances_pr['model'].isin(models_)]
    ax0 = sns.barplot(x='model',hue='features',y=0,data=performances_pr_[performances_pr_['features'].isin(order)],
                      ci=100-(0.05/len(order)),ax=axes[col],hue_order=order,order=models_[:-1])
    ax0.set_ylabel('AUPRC')
    if col==2:
        handles,_ = axes[col].get_legend_handles_labels()
        ax0.legend(bbox_to_anchor=(1,1),labels=['no-skill','genetics+family','lifestyle','blood','prodromal symptoms',
                                                'all accelerometer','combined'],#,'stacked'],
                  handles=handles,title='modality')
    else:
        ax0.legend([],[], frameon=False)
    ax0.set_xlabel('')
    ax0.set_xticklabels(label,rotation=0)
    ax0.set_title(title)
    #ax0.set_ylim(0,1)
    ax0.text(-0.1, 1.1, subplotlabelsA[col], transform=ax0.transAxes, 
            size=20, weight='bold')
plt.tight_layout()
plt.savefig(f'{image_path}/performance_barplot_stackedNoPD_pdprod_auprc.pdf',dpi=300,bbox_inches='tight')
plt.savefig(f'{image_path}/performance_barplot_stackedNoPD_pdprod_auprc.png',dpi=300,bbox_inches='tight')

In [ ]:
order = ['intercept','genetics+family','lifestyle_nofam','blood','prodromalsigns_beforePD','all_acc_features',
         'all_acc_features+blood+lifestyle+genetics+prodromalsigns_beforePD']
labels = [['Diagnosed','Prodromal','Diagnosed &\nProdromal'],
                         ['Diagnosed','Prodromal','Diagnosed &\nProdromal'],['Diagnosed','Prodromal','Diagnosed &\nProdromal']]
titles = ['control = matched unaffected','control = unaffected', 'control = general population']
fig,axes = plt.subplots(nrows=2,ncols=3,figsize=(30,7))
plots.plot_context()
sns.set_style('whitegrid')

subplotlabelsA = ['A','C','E']
subplotlabelsB = ['B','D','F']

for col,control,label,title in zip(np.arange(3),['[d|D]HC','allHC','Population'],labels,titles):
    r = re.compile(f'.*{control}.*')
    models_ = list(filter(r.match, models))
    performances_ = performances[performances['model'].isin(models_)]#performances['model'].str.contains(control)]
    ax0 = sns.barplot(x='model',hue='features',y=0,data=performances_[performances_['features'].isin(order)],ci=100-(0.05/len(order)),
                      ax=axes[0,col],hue_order=order,order=models_)
    ax0.set_ylabel('AUROC')
    if col==2:
        handles,_ = axes[0,col].get_legend_handles_labels()
        ax0.legend(bbox_to_anchor=(1,1),labels=['no-skill','genetics+family','lifestyle','blood','prodromal symptoms',
                                                'all accelerometer','combined'],#,'stacked'],
                  handles=handles,title='modality')
    else:
        ax0.legend([],[], frameon=False)
    ax0.set_xlabel('')
    ax0.set_xticklabels(label,rotation=0)
    ax0.set_title(title)
    ax0.set_ylim(0,1)
    ax0.text(-0.1, 1.1, subplotlabelsA[col], transform=ax0.transAxes, 
            size=20, weight='bold')
    
    performances_pr_ = performances_pr[performances_pr['model'].str.contains(control)]
    ax1 = sns.barplot(x='model',hue='features',y=0,data=performances_pr_[performances_pr_['features'].isin(order)],
                      ci=100-(0.05/len(order)),ax=axes[1,col],hue_order=order,order=models_)
    ax1.set_ylabel('AUPRC')
    ax1.set_xticklabels(label,rotation=0)
    ax1.set_xlabel('')
    ax1.legend([],[], frameon=False)
    ax1.set_ylim(0,1)
    ax1.text(-0.1, 1.1, subplotlabelsB[col], transform=ax1.transAxes, 
            size=20, weight='bold')
plt.tight_layout()
#plt.savefig(f'{image_path}/performance_barplot_stackedNoPD_allmodels_split.pdf',dpi=300,bbox_inches='tight')
#plt.savefig(f'{image_path}/performance_barplot_stackedNoPD_allmodels_split.png',dpi=300,bbox_inches='tight')

In [ ]:
comparison = performances[performances['features'].isin(np.hstack([modalities,modalities2,modalities3,'intercept']))]
comparison_pr = performances_pr[performances_pr['features'].isin(np.hstack([modalities,modalities2,modalities3,'intercept']))]

In [ ]:
def get_comp(comparison):
    compare = []
    for m,m2,m3 in zip(modalities,modalities2,modalities3):
        c = comparison[comparison['features'].isin(['intercept',m,m2,m3])]
        c['kind'] = m
        c.loc[c['features']=='intercept','modality'] = 'no-skill'
        c.loc[c['features']==m,'modality'] = 'single'
        c.loc[c['features']==m2,'modality'] = 'combined'
        c.loc[c['features']==m3,'modality'] = 'stacked'
        compare.append(c)
    compare = pd.concat(compare)
    return compare
    
compare = get_comp(comparison)
compare_pr = get_comp(comparison_pr)

In [ ]:
hue_order = ['no-skill','single','combined']#,'stacked']
labels = ['genetics+family','blood','lifestyle_nofam','prodromalsigns_beforePD','all_acc_features']
label_names = ['genetics+\nfamily','blood','lifestyle','prodromal\nsymptoms','all\naccelerometry']
titles = ['control = matched unaffected','control = all unaffected', 'control = general population']
models = ['diag_PDHC','diag_ProdHC','diag_PDProdHC','diag_PDallHC','diag_ProdallHC','diag_PDProdallHC','diag_PDPopulationNoProd','diag_ProdPopulationNoPD',
         'diag_PDProdPopulation']
for comp,auc in zip([compare,compare_pr],['auroc','auprc']):

    statistics = pd.DataFrame(columns=pd.MultiIndex.from_product([models,['T','dof','alternative','p-value','CI95%',
                                                                   'cohen-d','BF10','power']],names=['model','statistic']),
                           index=pd.MultiIndex.from_product([labels,hue_order],names=['kind','modality']))
    pvalues = pd.DataFrame(columns=pd.MultiIndex.from_product([models,['T','dof','alternative','p-value','CI95%',
                                                                   'cohen-d','BF10','power']],names=['model','statistic']),
                           index=pd.MultiIndex.from_product([labels,hue_order],names=['kind','modality']))
    for i,f1 in enumerate(labels):
        for f2,f3 in zip(['no-skill','single'],['single','combined']):
                for model in models:
                    c = comp[comp['model']==model]
                    t,p = stats.ttest_ind(c.loc[np.logical_and(c['kind']==f1,
                                                                             c['modality']==f2),0],
                                         c.loc[np.logical_and(c['kind']==f1,
                                                                            c['modality']==f3),0])
                    #statistics.loc[(f1,f2),(model,slice(None))] = pg.ttest(c.loc[np.logical_and(c['kind']==f1,
                    #                                    c['modality']==f2),0],
                    #c.loc[np.logical_and(c['kind']==f1,
                    #                                   c['modality']==f3),0]).values
                    pvalues.loc[(f1,f3),(model,'p-value')] = p
                    pvalues.loc[(f1,f3),(model,'T')] = t
    pvalues = pvalues.dropna(axis='rows',how='all')
    #statistics = statistics.dropna(axis='rows',how='all')
    # get pvalues that are significant
    keep = pvalues.loc[:,(slice(None),'p-value')]<(0.05/pvalues.shape[0])
    pvalues_sig = pvalues[keep]
    
    #statistics = statistics.reset_index()
    #statistics['modality'] = statistics['modality'].replace(['no-skill','single'],['no-skill vs single','single vs combined'])
    #statistics.set_index(['kind','modality']).to_csv(f'{data_path}/{auc}.csv')
    thresh = 0.05/(len(labels)*(len(hue_order)-1))
    annot = [[5e-6, "****"], [5e-5, "***"], [5e-4, "**"], [thresh, "*"], [1, "ns"]]
    
    for diags,diagname in zip([['diag_ProdHC','diag_ProdallHC','diag_ProdPopulationNoPD'],['diag_PDHC','diag_PDallHC','diag_PDPopulationNoProd'],
                  ['diag_PDProdHC','diag_PDProdallHC','diag_PDProdPopulation']],['prod','pd','pdprod']):
        fig,axes = plt.subplots(nrows=3,ncols=1,figsize=(13,10))
        plots.plot_context()
        sns.set_style('whitegrid')

        subplotlabelsA = ['A','B','C','D','E']

        for col,control,title in zip(np.arange(3),diags,titles):
            performances_pr_ = comp[comp['model']==control]
            ax1 = sns.barplot(x='kind',hue='modality',y=0,data=performances_pr_,
                              ci=100-(0.05/(len(labels)*2)),ax=axes[col],hue_order=hue_order,order=labels)
            box_pairs = []
            sig = pvalues_sig[control].dropna(axis='rows',how='all',subset=['p-value'])
            for key,s in sig.iterrows():
                if key[1] == 'single':
                    box_pairs.append(((key[0],'no-skill'),(key[0],'single')))
                elif key[1] == 'combined' or key[1] == 'stacked':
                    box_pairs.append(((key[0],'single'),(key[0],key[1])))
            if len(box_pairs)>0:
                ax, test_results = add_stat_annotation(axes[col], data=performances_pr_, x='kind', y=0, hue='modality',hue_order=hue_order,
                                                       order=labels,test='t-test_ind', text_format='star', loc='outside', verbose=2,box_pairs=box_pairs,
                                                        comparisons_correction=None,pvalue_thresholds=annot)
            ax1.set_ylabel(auc.upper())
            if col==0:
                handles,_ = axes[col].get_legend_handles_labels()
                ax1.legend(bbox_to_anchor=(1,1),labels=['no-skill','single','combined','stacked'],handles=handles,title='modality')
            else:
                ax1.legend([],[], frameon=False)
            ax1.set_xticklabels(label_names,rotation=0)
            ax1.set_xlabel('')
            ax1.set_title(title,y=1.45)
            ax1.text(-0.1, 1.1, subplotlabelsA[col], transform=ax1.transAxes, 
                    size=20, weight='bold')
        plt.tight_layout()
        #plt.savefig(f'{image_path}/performance_barplot_stackedNoPD_{diagname}models_{auc}.pdf',dpi=300,bbox_inches='tight')
        #plt.savefig(f'{image_path}/performance_barplot_stackedNoPD_{diagname}models_{auc}.png',dpi=300,bbox_inches='tight')

# Supplemental Figures

ROC and PRC plots

In [ ]:
def plot_rocauc_test(performance,feature=['PRS'],fnames=['genetic'],savefig=[]):
    colors = sns.color_palette("colorblind", 10)
    linestyles = ['-','--']
    perf = performance.loc[(slice(None),feature),:]
    f, ax = plt.subplots(figsize=(6, 6))
    plots.plot_context()
    for i,f in enumerate(feature):
        for j,model in enumerate(np.unique(perf.index.get_level_values(level=0))):
                auc = perf.loc[(model,f),'AUROC']
                ax.plot(perf.loc[(model,f),'fpr'],perf.loc[(model,f),'tpr'],
                         label=f'{fnames[i]} AUROC: {auc:.2f}',color=colors[i],ls=linestyles[0])
    ax.set_xlabel('false positive rate')
    ax.set_ylabel('true positive rate')
    ax.legend(bbox_to_anchor=[1,1])
    ax.plot([0, 1], [0, 1], transform=ax.transAxes,color='k',linestyle='--')
    if savefig:
        plt.savefig(savefig,bbox_inches='tight',dpi=200)
    plt.show()

In [ ]:
models = ['diag_PDProdPopulation']
kinds = ['pop']
features = ['all_acc_features','acc']
names = ['Population vs PD']
fnames = ['average acceleration','all accelerometer']
performance = pd.DataFrame(index=pd.MultiIndex.from_product([names,features],names=['model','features']))

for kind,model,name in zip(kinds,models,names):
    for feature in features:
        path = f'{data_path}/{feature}/{kind}modelsNoPD_perf_stacked_allHC.csv'
        perf = pd.read_csv(path,index_col=[0,1],header=[0,1])
        perf = perf['test']
        perf_a = perf.loc[model,'fpr'].dropna()
        performance.loc[(name,feature),'fpr'] = ','.join(perf_a.values[0].split())
        perf_p = perf.loc[model,'tpr'].dropna()
        performance.loc[(name,feature),'tpr'] = ','.join(perf_p.values[0].split())
        perf_p = perf.loc[model,'ROCAUC'].dropna()
        print(perf_p.mean())
        performance.loc[(name,feature),'AUROC'] = perf_p.values[0]
performance['fpr'] = performance['fpr'].apply(literal_eval)
performance['tpr'] = performance['tpr'].apply(literal_eval)

In [ ]:
plot_rocauc_test(performance,feature=['acc','all_acc_features'],fnames=fnames,
                 savefig=[])#'/scratch/c.c21013066/images/ukbiobank/accelerometer/models/acc_features_PDPopulation_fold0.png')

In [ ]:
# recompute tpr,fpr,recall, precision and save correctly
models = ['diag_PDHC','diag_ProdHC','diag_PDProdHC','diag_PDallHC','diag_ProdallHC','diag_PDProdallHC','diag_PDPopulationNoProd','diag_ProdPopulationNoPD',
          'diag_PDProdPopulation']
models0 = ['diag_PDHC','diag_ProdHC','diag_PDProdHC','diag_PDHC','diag_ProdHC','diag_PDProdHC','diag_PDPopulationNoProd','diag_ProdPopulationNoPD',
          'diag_PDProdPopulation']
kinds = ['HC','HC','HC','allHC','allHC','allHC','pop','pop','pop']
features = ['intercept','covariates','PRS','genetics+family','lifestyle_nofam','blood','prodromalsigns_beforePD','acc','all_acc_features',
            'genetics+family+all_acc_features','lifestyle+all_acc_features','blood+all_acc_features','prodromalsigns_beforePD+all_acc_features',
            'all_acc_features+blood+lifestyle+genetics+prodromalsigns_beforePD']
stacked_features = ['genetics+family+all_acc_features','lifestyle+all_acc_features','blood+all_acc_features','prodromalsigns_beforePD+all_acc_features',
            'all_acc_features+blood+lifestyle+genetics+prodromalsigns_beforePD']
stacked_features_names = [f'{n}_stacked' for n in stacked_features]

curves = pd.DataFrame(index=pd.MultiIndex.from_product([models,np.hstack([features,stacked_features_names])],names=['model','features']),
                            columns=pd.MultiIndex.from_product([['tpr','fpr','recall','precision','auroc','auprc'],np.arange(5)]))
lengths = [200,200,200,6000,6000,6000,7000,7000,7000]
length=200

for mod,modname,kd,length in zip([models[:3],models[3:6],models[6:]],[models0[:3],models0[3:6],models0[6:]],
                                   [kinds[:3],kinds[3:6],kinds[6:]],[200,6000,7000]):

    for i,feature in enumerate(features):
        curve = pd.DataFrame(index=pd.MultiIndex.from_product([mod,np.arange(5)],names=['model','cv']),
                                columns=pd.MultiIndex.from_product([['tpr','fpr','recall','precision'],np.arange(length)]))
        for (m,kind),modelname,model in zip(enumerate(kd),mod,modname):
            if 'stacked' in feature:
                name = modelname.replace('diag_','')
                # fname = stacked_features[i]
                # stacked = pd.read_csv(f'{data_path}/{fname}/stacked_perf.csv',index_col=[0,1],header=[0,1])
                # stacked = stacked['test']
                # performances.loc[(modelname,feature),:] = stacked.loc[name,'AUROC'].dropna().values
                # performances_pr.loc[(modelname,feature),:] = stacked.loc[name,'AUPRC'].dropna().values
                # curves.loc[(modelname,feature),('fpr',slice(None))] = stacked.loc[name,'fpr'].dropna().values
                # curves.loc[(modelname,feature),('tpr',slice(None))] = stacked.loc[name,'tpr'].dropna().values
                # curves.loc[(modelname,feature),('precision',slice(None))] = stacked.loc[name,'precision'].dropna().values
                # curves.loc[(modelname,feature),('recall',slice(None))] = stacked.loc[name,'recall'].dropna().values
            else:
                if kind == 'HC':
                    path = f'{data_path}/{feature}/HCmodels_pred_stacked_matched.csv'
                    true = pd.read_csv(f'/scratch/c.c21013066/data/ukbiobank/merged_data/unaffectedNoOsteoMatchedHC.csv').set_index('eid')
                elif kind == 'allHC':
                    path = f'{data_path}/{feature}/HCmodels_pred_stacked_allHC.csv'
                    true = pd.read_csv(f'/scratch/c.c21013066/data/ukbiobank/merged_data/unaffectedNoOsteoAllHC.csv').set_index('eid')
                elif kind == 'pop':
                    path = f'{data_path}/{feature}/popmodelsNoPD_pred_stacked_allHC.csv'
                    true = pd.read_csv(f'/scratch/c.c21013066/data/ukbiobank/merged_data/populationNoOsteoAllHC.csv').set_index('eid')
                perf = pd.read_csv(path,index_col=[0,1],header=[0,1])
                perf = perf.loc[:,(slice(None),'test')].droplevel(level=1,axis=1).loc[:,[model]].dropna()
                for cv in np.arange(5):
                    perf_cv = perf.loc[(slice(None),cv),:]
                    m = pd.merge(perf_cv,true[[model]],right_index=True,left_index=True,how='left')
                    nona = m.dropna(how='any',axis='rows')
                    pre,rec,_ = metrics.precision_recall_curve(nona.iloc[:,1],nona.iloc[:,0])
                    fpr, tpr, _ = metrics.roc_curve(nona.iloc[:,1],nona.iloc[:,0])
                    auc = metrics.roc_auc_score(nona.iloc[:,1],nona.iloc[:,0])
                    auprc = metrics.average_precision_score(nona.iloc[:,1],nona.iloc[:,0])
                    curves.loc[(modelname,feature),('tpr',cv)] = tpr
                    curves.loc[(modelname,feature),('fpr',cv)] = fpr
                    curves.loc[(modelname,feature),('precision',cv)] = pre
                    curves.loc[(modelname,feature),('recall',cv)] = rec
                    curves.loc[(modelname,feature),('auroc',cv)] = auc
                    curves.loc[(modelname,feature),('auprc',cv)] = auprc
                    curve.loc[(modelname,cv),('tpr',slice(None))] = np.append(tpr, np.zeros(length-len(tpr)) + np.nan)
                    curve.loc[(modelname,cv),('fpr',slice(None))]= np.append(fpr, np.zeros(length-len(fpr)) + np.nan)
                    curve.loc[(modelname,cv),('recall',slice(None))]= np.append(rec, np.zeros(length-len(rec)) + np.nan)
                    curve.loc[(modelname,cv),('precision',slice(None))]= np.append(pre, np.zeros(length-len(pre)) + np.nan)
        if kind == 'HC':
            curve.to_csv(f'{data_path}/{feature}/HCmodels_curves_stacked_matched.csv')
        elif kind == 'allHC':
            curve.to_csv(f'{data_path}/{feature}/HCmodels_curves_stacked_allHC.csv')
        if kind == 'pop':
            curve.to_csv(f'{data_path}/{feature}/popmodels_curves_stacked_allHC.csv')

In [ ]:
# instead combine all predictions and compute PRC over that
models = ['diag_PDHC','diag_ProdHC','diag_PDProdHC','diag_PDallHC','diag_ProdallHC','diag_PDProdallHC','diag_PDPopulationNoProd','diag_ProdPopulationNoPD',
          'diag_PDProdPopulation']
models0 = ['diag_PDHC','diag_ProdHC','diag_PDProdHC','diag_PDHC','diag_ProdHC','diag_PDProdHC','diag_PDPopulationNoProd','diag_ProdPopulationNoPD',
          'diag_PDProdPopulation']
kinds = ['HC','HC','HC','allHC','allHC','allHC','pop','pop','pop']
features = ['intercept','covariates','PRS','genetics+family','lifestyle_nofam','blood','prodromalsigns_beforePD','acc','all_acc_features',
            'genetics+family+all_acc_features','lifestyle+all_acc_features','blood+all_acc_features','prodromalsigns_beforePD+all_acc_features',
            'all_acc_features+blood+lifestyle+genetics+prodromalsigns_beforePD']
stacked_features = ['genetics+family+all_acc_features','lifestyle+all_acc_features','blood+all_acc_features','prodromalsigns_beforePD+all_acc_features',
            'all_acc_features+blood+lifestyle+genetics+prodromalsigns_beforePD']
stacked_features_names = [f'{n}_stacked' for n in stacked_features]

curves_combined = pd.DataFrame(index=pd.MultiIndex.from_product([models,np.hstack([features,stacked_features_names])],names=['model','features']),
                            columns=pd.MultiIndex.from_product([['tpr','fpr','recall','precision','auroc','auprc'],['mean','std']]))

for kind,modelname,model in zip(kinds,models,models0):
    for i,feature in enumerate(np.hstack([stacked_features_names,features])):
        if 'stacked' in feature:
            name = modelname.replace('diag_','')
            # fname = stacked_features[i]
            # stacked = pd.read_csv(f'{data_path}/{fname}/stacked_perf.csv',index_col=[0,1],header=[0,1])
            # stacked = stacked['test']
            # performances.loc[(modelname,feature),:] = stacked.loc[name,'AUROC'].dropna().values
            # performances_pr.loc[(modelname,feature),:] = stacked.loc[name,'AUPRC'].dropna().values
            # curves.loc[(modelname,feature),('fpr',slice(None))] = stacked.loc[name,'fpr'].dropna().values
            # curves.loc[(modelname,feature),('tpr',slice(None))] = stacked.loc[name,'tpr'].dropna().values
            # curves.loc[(modelname,feature),('precision',slice(None))] = stacked.loc[name,'precision'].dropna().values
            # curves.loc[(modelname,feature),('recall',slice(None))] = stacked.loc[name,'recall'].dropna().values
        else:
            if kind == 'HC':
                path = f'{data_path}/{feature}/HCmodels_pred_stacked_matched.csv'
                true = pd.read_csv(f'/scratch/c.c21013066/data/ukbiobank/merged_data/unaffectedNoOsteoMatchedHC.csv').set_index('eid')
            elif kind == 'allHC':
                path = f'{data_path}/{feature}/HCmodels_pred_stacked_allHC.csv'
                true = pd.read_csv(f'/scratch/c.c21013066/data/ukbiobank/merged_data/unaffectedNoOsteoAllHC.csv').set_index('eid')
            elif kind == 'pop':
                path = f'{data_path}/{feature}/popmodelsNoPD_pred_stacked_allHC.csv'
                true = pd.read_csv(f'/scratch/c.c21013066/data/ukbiobank/merged_data/populationNoOsteoAllHC.csv').set_index('eid')
            perf = pd.read_csv(path,index_col=[0,1],header=[0,1])
            perf = perf.loc[:,(slice(None),'test')].droplevel(level=1,axis=1).loc[:,model].dropna()
            perf = perf.droplevel(level=1,axis=0)

            m = pd.merge(perf,true[[model]],right_index=True,left_index=True,how='left')
            nona = m.dropna(how='any',axis='rows')
            pre,rec,_ = metrics.precision_recall_curve(nona.iloc[:,1],nona.iloc[:,0])
            fpr, tpr, _ = metrics.roc_curve(nona.iloc[:,1],nona.iloc[:,0])
            auc = metrics.roc_auc_score(nona.iloc[:,1],nona.iloc[:,0])
            auprc = metrics.average_precision_score(nona.iloc[:,1],nona.iloc[:,0])
            curves_combined.loc[(modelname,feature),('tpr','mean')] = tpr
            curves_combined.loc[(modelname,feature),('fpr','mean')] = fpr
            curves_combined.loc[(modelname,feature),('precision','mean')] = pre
            curves_combined.loc[(modelname,feature),('recall','mean')] = rec
            curves_combined.loc[(modelname,feature),('auroc','mean')] = auc
            curves_combined.loc[(modelname,feature),('auprc','mean')] = auprc

In [ ]:
curves_combined.to_csv(f'{data_path}/curves_combined.csv')

In [ ]:
def string_to_array(st):
    try:
        return np.array(list(map(float,st[1:-1].split())))
    except:
        st = st.replace('...','')
        return np.array(list(map(float,st[1:-1].split())))

def plot_AUROC_CV(curves,pf,ax,features=[],fnames=[],label='',control=''):
    #ax.plot([0, 1], [0, 1], "k--", label="chance level (AUC = 0.5)")
    if len(features) == 0:
        features = pf.index
    if len(fnames) == 0:
        fnames = features
    for (i,feature),fname in zip(enumerate(features),fnames):
        c = curves.loc[feature,:].copy()
        p = pf.loc[feature,:].copy()
        
        i_tprs = []
        mean_fpr = np.linspace(0, 1, 100)
        for cv in range(5):
            interp_tpr = np.interp(mean_fpr, c.loc['fpr',cv], c.loc['tpr',cv])
            interp_tpr[0] = 0.0
            i_tprs.append(interp_tpr)

        mean_tpr = np.mean(i_tprs, axis=0)
        mean_tpr[-1] = 1.0

        ax.plot(
            mean_fpr,
            mean_tpr,
            label=f"{fname} (AUROC = {p.loc['AUROC','mean']:.2f} $\pm$  {p.loc['AUROC','std']:.2f})",
            lw=2,
            alpha=0.8,
        )

        std_tpr = np.std(i_tprs, axis=0)
        tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
        tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
        if i == 0:
            ax.fill_between(
                mean_fpr,
                tprs_lower,
                tprs_upper,
                color=ax.get_lines()[-1].get_c(),
                alpha=0.2,
                label=r"$\pm$ 1 std",
            )
        else:
            ax.fill_between(
            mean_fpr,
            tprs_lower,
            tprs_upper,
            color=ax.get_lines()[-1].get_c(),
            alpha=0.2,
        )

        ax.set(
            #xlim=[-0.05, 1.05],
            #ylim=[-0.05, 1.05],
            xlabel="False Positive Rate",
            ylabel="True Positive Rate",
            #title=f'{control} vs {label}'
        )
        ax.axis("square")
    ax.legend(bbox_to_anchor=[1,1],title=f'{label} vs\n{control}')
    
def plot_AUPRC_CV(curves,pf,ax,features=[],fnames=[],label='',control='',noskill=True):
    #ax.plot([0, 1], [0, 1], "k--", label="chance level (AUC = 0.5)")
    if len(features) == 0:
        features = pf.index
    if len(fnames) == 0:
        fnames = features
    for (i,feature),fname in zip(enumerate(features),fnames):
        c = curves.loc[feature,:].copy()
        p = pf.loc[feature,:].copy()
        
        i_tprs = []
        mean_fpr = np.linspace(0, 1, 100)
        for cv in range(5):
#             ax.plot(
#                 string_to_array(c.loc['recall',cv]),
#                 string_to_array(c.loc['precision',cv]),
#                 label=f"{fname}",
#                 lw=2,
#                 alpha=0.8,
#             )
            interp_tpr = np.interp(mean_fpr, c.loc['recall',cv][::-1], c.loc['precision',cv][::-1])
            interp_tpr[0] = 0.0
            i_tprs.append(interp_tpr)

        mean_tpr = np.mean(i_tprs, axis=0)
        mean_tpr[0] = 0.0
        #mean_tpr[-1] = 0.0
        if noskill and fname=='no-skill':
            ax.plot(
            [1,0],
            [mean_tpr[-1],mean_tpr[-1]],
            label=f"{fname} (AUPRC = {p.loc['AUPRC','mean']:.2f} $\pm$  {p.loc['AUPRC','std']:.2f})",
            lw=2,
            alpha=0.8,
            )

        else:
            ax.plot(
                mean_fpr,
                mean_tpr,
                label=f"{fname} (AUPRC = {p.loc['AUPRC','mean']:.2f} $\pm$  {p.loc['AUPRC','std']:.2f})",
                lw=2,
                alpha=0.8,
            )

            std_tpr = np.std(i_tprs, axis=0)
            tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
            tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
            if i == 0:
                ax.fill_between(
                    mean_fpr,
                    tprs_lower,
                    tprs_upper,
                    color=ax.get_lines()[-1].get_c(),
                    alpha=0.2,
                    label=r"$\pm$ 1 std",
                )
            else:
                ax.fill_between(
                mean_fpr,
                tprs_lower,
                tprs_upper,
                color=ax.get_lines()[-1].get_c(),
                alpha=0.2,
            )

        ax.set(
            #xlim=[-0.05, 1.05],
            #ylim=[-0.05, 1.05],
            xlabel="Recall",
            ylabel="Precision",
            #title=f'{control} vs {label}'
        )
        ax.axis("square")
    ax.legend(bbox_to_anchor=[1,1],title=f'{label} vs\n{control}')
    
    
def plot_AUPRC_CV_new(diag,pf,ax,features=[],fnames=[],label='',control='',noskill=True,diag_alt=''):
    #ax.plot([0, 1], [0, 1], "k--", label="chance level (AUC = 0.5)")
    if len(features) == 0:
        features = pf.index
    if len(fnames) == 0:
        fnames = features
    if control == 'matched unaffected controls':
        cont = 'HC'
        kind = 'matched'
    elif control == 'unaffected controls':
        cont = 'HC'
        kind = 'allHC'
    elif control == 'population':
        cont = 'pop'
        kind = 'allHC'
        
    for (i,feature),fname in zip(enumerate(features),fnames):
        curve = pd.read_csv(f'{data_path}/{feature}/{cont}models_curves_stacked_{kind}.csv',index_col=[0,1],header=[0,1])
        c = curve.loc[diag,:].copy()
        p = pf.loc[feature,:].copy()
        
        i_tprs = []
        mean_fpr = np.linspace(0, 1, 100)
        for cv in range(5):
#             ax.plot(
#                 string_to_array(c.loc['recall',cv]),
#                 string_to_array(c.loc['precision',cv]),
#                 label=f"{fname}",
#                 lw=2,
#                 alpha=0.8,
#             )
            interp_tpr = np.interp(mean_fpr, c.loc[cv,'recall'].dropna()[::-1], c.loc[cv,'precision'].dropna()[::-1])
            interp_tpr[0] = 0.0
            i_tprs.append(interp_tpr)

        mean_tpr = np.mean(i_tprs, axis=0)
        mean_tpr[0] = 0.0
        #mean_tpr[-1] = 0.0
        if noskill and fname=='no-skill':
            ax.plot(
            [1,0],
            [mean_tpr[-1],mean_tpr[-1]],
            label=f"{fname} (AUPRC = {p.loc['AUPRC','mean']:.2f} $\pm$  {p.loc['AUPRC','std']:.2f})",
            lw=2,
            alpha=0.8,
            )

        else:
            ax.plot(
                mean_fpr,
                mean_tpr,
                label=f"{fname} (AUPRC = {p.loc['AUPRC','mean']:.2f} $\pm$  {p.loc['AUPRC','std']:.2f})",
                lw=2,
                alpha=0.8,
            )

            std_tpr = np.std(i_tprs, axis=0)
            tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
            tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
            if i == 0:
                ax.fill_between(
                    mean_fpr,
                    tprs_lower,
                    tprs_upper,
                    color=ax.get_lines()[-1].get_c(),
                    alpha=0.2,
                    label=r"$\pm$ 1 std",
                )
            else:
                ax.fill_between(
                mean_fpr,
                tprs_lower,
                tprs_upper,
                color=ax.get_lines()[-1].get_c(),
                alpha=0.2,
            )

        ax.set(
            #xlim=[-0.05, 1.05],
            #ylim=[-0.05, 1.05],
            xlabel="Recall",
            ylabel="Precision",
            #title=f'{control} vs {label}'
        )
        ax.axis("square")
    ax.legend(bbox_to_anchor=[1,1],title=f'{label} vs\n{control}')
    
def plot_AUROC_CV_new(diag,pf,ax,features=[],fnames=[],label='',control='',noskill=True,diag_alt=''):
    #ax.plot([0, 1], [0, 1], "k--", label="chance level (AUC = 0.5)")
    if len(features) == 0:
        features = pf.index
    if len(fnames) == 0:
        fnames = features
    if control == 'matched unaffected controls':
        cont = 'HC'
        kind = 'matched'
    elif control == 'unaffected controls':
        cont = 'HC'
        kind = 'allHC'
    elif control == 'population':
        cont = 'pop'
        kind = 'allHC'
        
    for (i,feature),fname in zip(enumerate(features),fnames):
        print(fname,feature)
        curve = pd.read_csv(f'{data_path}/{feature}/{cont}models_curves_stacked_{kind}.csv',index_col=[0,1],header=[0,1])
        #try:
        c = curve.loc[diag,:].copy()
        #except:
        #    c = curve.loc[diag_alt,:].copy()
        p = pf.loc[feature,:].copy()
        
        i_tprs = []
        mean_fpr = np.linspace(0, 1, 100)
        for cv in range(5):
#             ax.plot(
#                 string_to_array(c.loc['recall',cv]),
#                 string_to_array(c.loc['precision',cv]),
#                 label=f"{fname}",
#                 lw=2,
#                 alpha=0.8,
#             )
            interp_tpr = np.interp(mean_fpr, c.loc[cv,'fpr'].dropna(), c.loc[cv,'tpr'].dropna())
            interp_tpr[0] = 0.0
            i_tprs.append(interp_tpr)

        mean_tpr = np.mean(i_tprs, axis=0)
        mean_tpr[-1] = 1.0
        #mean_tpr[-1] = 0.0
        ax.plot(
            mean_fpr,
            mean_tpr,
            label=f"{fname} (AUROC = {p.loc['AUROC','mean']:.2f} $\pm$  {p.loc['AUROC','std']:.2f})",
            lw=2,
            alpha=0.8,
        )

        std_tpr = np.std(i_tprs, axis=0)
        tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
        tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
        if i == 0:
            ax.fill_between(
                mean_fpr,
                tprs_lower,
                tprs_upper,
                color=ax.get_lines()[-1].get_c(),
                alpha=0.2,
                label=r"$\pm$ 1 std",
            )
        else:
            ax.fill_between(
            mean_fpr,
            tprs_lower,
            tprs_upper,
            color=ax.get_lines()[-1].get_c(),
            alpha=0.2,
        )

        ax.set(
            #xlim=[-0.05, 1.05],
            #ylim=[-0.05, 1.05],
            xlabel="False Positive Rate",
            ylabel="True Positive Rate",
            #title=f'{control} vs {label}'
        )
        ax.axis("square")
    ax.legend(bbox_to_anchor=[1,1],title=f'{label} vs\n{control}')

    
def plot_AUROC_concatCV(curves,pf,ax,features=[],fnames=[],label='',control=''):
    #ax.plot([0, 1], [0, 1], "k--", label="chance level (AUC = 0.5)")
    if len(features) == 0:
        features = pf.index
    if len(fnames) == 0:
        fnames = features
    for (i,feature),fname in zip(enumerate(features),fnames):
        c = curves.loc[feature,:].copy()
        p = pf.loc[feature,:].copy()

        ax.plot(
            c.loc['fpr','mean'],
            c.loc['tpr','mean'],
            label=f"{fname} (AUROC = {p.loc['AUROC','mean']:.2f} $\pm$  {p.loc['AUROC','std']:.2f})",
            lw=2,
            alpha=0.8,
        )

        ax.set(
            #xlim=[-0.05, 1.05],
            #ylim=[-0.05, 1.05],
            xlabel="False Positive Rate",
            ylabel="True Positive Rate",
            #title=f'{control} vs {label}'
        )
        ax.axis("square")
    ax.legend(bbox_to_anchor=[1,1],title=f'{label} vs\n{control}')
    
def plot_AUPRC_concatCV(curves,pf,ax,features=[],fnames=[],label='',control='',noskill=True):
    #ax.plot([0, 1], [0, 1], "k--", label="chance level (AUC = 0.5)")
    if len(features) == 0:
        features = pf.index
    if len(fnames) == 0:
        fnames = features
    for (i,feature),fname in zip(enumerate(features),fnames):
        c = curves.loc[feature,:].copy()
        p = pf.loc[feature,:].copy()
        if noskill and fname=='no-skill':
            ax.plot(
            [1,0],
            [c.loc['precision','mean'][0],c.loc['precision','mean'][0]],
            label=f"{fname} (AUPRC = {p.loc['AUPRC','mean']:.2f} $\pm$  {p.loc['AUPRC','std']:.2f})",
            lw=2,
            alpha=0.8,
            )

        else:
            ax.plot(
                c.loc['recall','mean'],
                c.loc['precision','mean'],
                label=f"{fname} (AUPRC = {p.loc['AUPRC','mean']:.2f} $\pm$  {p.loc['AUPRC','std']:.2f})",
                lw=2,
                alpha=0.8,
            )
        ax.set(
            xlim=[-0.05, 1.05],
            ylim=[-0.05, 1.05],
            xlabel="Recall",
            ylabel="Precision",
            #title=f'{control} vs {label}'
        )
        ax.axis("square")
    ax.legend(bbox_to_anchor=[1,1],title=f'{label} vs\n{control}')

In [ ]:
saves = ['AUPRC_matchedHC','AUPRC_allHC','AUPRC_population']
controls = ['matched unaffected controls','unaffected controls','population']
diagss = [['diag_PDHC','diag_ProdHC','diag_PDProdHC'],['diag_PDallHC','diag_ProdallHC','diag_PDProdallHC'],
          ['diag_PDPopulationNoProd','diag_ProdPopulationNoPD','diag_PDProdPopulation']]
labels = ['Diagnosed','Prodromal','Diagnosed & Prodromal']
subfigs = ['A','B','C']

for save,control,diags in zip(saves,controls,diagss):
    fig,axes = plt.subplots(nrows=3,ncols=1,figsize=(15,25))
    for diag,ax,label,subfig in zip(diags,axes.flatten(),labels,subfigs):
        print(diag,control)
        plot_AUPRC_concatCV(curves_combined.loc[(diag,slice(None)),:].droplevel(level=0,axis=0),pf.loc[(diag,slice(None)),:].droplevel(level=0,axis=0),
                      features=['intercept','genetics+family','lifestyle_nofam','blood','prodromalsigns_beforePD','all_acc_features',
                                 'all_acc_features+blood+lifestyle+genetics+prodromalsigns_beforePD'],
                      fnames=['no-skill','genetics+family','lifestyle','blood','prodromal symptoms',
                                                'all accelerometer','combined'],ax=ax,label=label,control=control)
        ax.text(-0.1, 1, subfig, transform=ax.transAxes, 
                size=20, weight='bold')
    if save:
        plt.savefig(f'{image_path}/{save}_concat.png',dpi=300,bbox_inches='tight')
        plt.savefig(f'{image_path}/{save}_concat.pdf',dpi=300,bbox_inches='tight')

In [ ]:
saves = ['AUROC_matchedHC','AUROC_allHC','AUROC_population']
controls = ['matched unaffected controls','unaffected controls','population']
diagss = [['diag_PDHC','diag_ProdHC','diag_PDProdHC'],['diag_PDallHC','diag_ProdallHC','diag_PDProdallHC'],['diag_PDPopulationNoProd','diag_ProdPopulationNoPD','diag_PDProdPopulation']]
labels = ['Diagnosed','Prodromal','Diagnosed & Prodromal']
subfigs = ['A','B','C']

for save,control,diags in zip(saves,controls,diagss):
    fig,axes = plt.subplots(nrows=3,ncols=1,figsize=(15,25))
    for diag,ax,label,subfig in zip(diags,axes.flatten(),labels,subfigs):
        print(diag,control)
        plot_AUROC_concatCV(curves_combined.loc[(diag,slice(None)),:].droplevel(level=0,axis=0),pf.loc[(diag,slice(None)),:].droplevel(level=0,axis=0),
                      features=['intercept','genetics+family','lifestyle_nofam','blood','prodromalsigns_beforePD','all_acc_features',
                                 'all_acc_features+blood+lifestyle+genetics+prodromalsigns_beforePD'],
                      fnames=['no-skill','genetics+family','lifestyle','blood','prodromal symptoms',
                                                'all accelerometer','combined'],ax=ax,label=label,control=control)
        ax.text(-0.1, 1, subfig, transform=ax.transAxes, 
                size=20, weight='bold')
    if save:
        plt.savefig(f'{image_path}/{save}_concat.png',dpi=300,bbox_inches='tight')
        plt.savefig(f'{image_path}/{save}_concat.pdf',dpi=300,bbox_inches='tight')

In [ ]:
saves = ['AUROC_matchedHC','AUROC_allHC','AUROC_population']
controls = ['matched unaffected controls','unaffected controls','population']
diagss = [['diag_PDHC','diag_ProdHC','diag_PDProdHC'],['diag_PDallHC','diag_ProdallHC','diag_PDProdallHC'],
          ['diag_PDPopulationNoProd','diag_ProdPopulationNoPD','diag_PDProdPopulation']]
diagss_alt = [['diag_PDHC','diag_ProdHC','diag_PDProdHC'],['diag_PDHC','diag_ProdHC','diag_PDProdHC'],
          ['diag_PDPopulationNoProd','diag_ProdPopulationNoPD','diag_PDProdPopulation']]
labels = ['Diagnosed','Prodromal','Diagnosed &\nProdromal']
subfigs = ['A','B','C']

for save,control,diags,diags_alt in zip(saves,controls,diagss,diagss_alt):
    fig,axes = plt.subplots(nrows=3,ncols=1,figsize=(15,25))
    plots.plot_context()
    for diag,diag_alt,ax,label,subfig in zip(diags,diags_alt,axes.flatten(),labels,subfigs):
        print(diag,control)
        plot_AUROC_CV(curves.loc[(diag,slice(None)),:].droplevel(level=0,axis=0),pf.loc[(diag,slice(None)),:].droplevel(level=0,axis=0),ax=ax,
                      features=['intercept','genetics+family','lifestyle_nofam','blood','prodromalsigns_beforePD','all_acc_features',
                                 'all_acc_features+blood+lifestyle+genetics+prodromalsigns_beforePD'],
                      fnames=['no-skill','genetics+family','lifestyle','blood','prodromal symptoms',
                                                'all accelerometer','combined'],label=label,control=control)
        ax.text(-0.1, 1, subfig, transform=ax.transAxes, 
                size=20, weight='bold')

    if save:
        plt.savefig(f'{image_path}/{save}.png',dpi=300,bbox_inches='tight')
        plt.savefig(f'{image_path}/{save}.pdf',dpi=300,bbox_inches='tight')

In [ ]:
saves = ['AUPRC_matchedHC','AUPRC_allHC','AUPRC_population']
controls = ['matched unaffected controls','unaffected controls','population']
diagss = [['diag_PDHC','diag_ProdHC','diag_PDProdHC'],['diag_PDallHC','diag_ProdallHC','diag_PDProdallHC'],['diag_PDPopulationNoProd','diag_ProdPopulationNoPD','diag_PDProdPopulation']]
labels = ['Diagnosed','Prodromal','Diagnosed & Prodromal']
subfigs = ['A','B','C']

for save,control,diags,diags_alt in zip(saves,controls,diagss,diagss_alt):
    fig,axes = plt.subplots(nrows=3,ncols=1,figsize=(15,25))
    plots.plot_context()
    for diag,diag_alt,ax,label,subfig in zip(diags,diags_alt,axes.flatten(),labels,subfigs):
        print(diag,control)
        plot_AUPRC_CV(curves.loc[(diag,slice(None)),:].droplevel(level=0,axis=0),pf.loc[(diag,slice(None)),:].droplevel(level=0,axis=0),ax=ax,
                      features=['intercept','genetics+family','lifestyle_nofam','blood','prodromalsigns_beforePD','all_acc_features',
                                 'all_acc_features+blood+lifestyle+genetics+prodromalsigns_beforePD'],
                      fnames=['no-skill','genetics+family','lifestyle','blood','prodromal symptoms',
                                                'all accelerometer','combined'],label=label,control=control)
        ax.text(-0.1, 1, subfig, transform=ax.transAxes, 
                size=20, weight='bold')

    if save:
        plt.savefig(f'{image_path}/{save}.png',dpi=300,bbox_inches='tight')
        plt.savefig(f'{image_path}/{save}.pdf',dpi=300,bbox_inches='tight')

In [ ]:
saves = ['AUPRC_matchedHC','AUPRC_allHC','AUPRC_population']
controls = ['matched unaffected controls','unaffected controls','population']
diagss = [['diag_PDHC','diag_ProdHC','diag_PDProdHC'],['diag_PDallHC','diag_ProdallHC','diag_PDProdallHC'],['diag_PDPopulationNoProd','diag_ProdPopulationNoPD','diag_PDProdPopulation']]
labels = ['Diagnosed','Prodromal','Diagnosed & Prodromal']
subfigs = ['A','B','C']

for save,control,diags in zip(saves,controls,diagss):
    fig,axes = plt.subplots(nrows=3,ncols=1,figsize=(15,25))
    for diag,ax,label,subfig in zip(diags,axes.flatten(),labels,subfigs):
        print(diag,control)
        plot_AUPRC_CV(curves.loc[(diag,slice(None)),:].droplevel(level=0,axis=0),pf.loc[(diag,slice(None)),:].droplevel(level=0,axis=0),
                      features=['intercept','genetics+family','lifestyle_nofam','blood','prodromalsigns_beforePD','all_acc_features',
                                 'all_acc_features+blood+lifestyle+genetics+prodromalsigns_beforePD'],
                      fnames=['no-skill','genetics+family','lifestyle','blood','prodromal symptoms',
                                                'all accelerometer','combined'],ax=ax,label=label,control=control)
        ax.text(-0.1, 1, subfig, transform=ax.transAxes, 
                size=20, weight='bold')

    if save:
        plt.savefig(f'{image_path}/{save}.png',dpi=300,bbox_inches='tight')
        plt.savefig(f'{image_path}/{save}.pdf',dpi=300,bbox_inches='tight')

In [ ]:
saves = ['AUROC_matchedHC','AUROC_allHC','AUROC_population']
controls = ['matched unaffected controls','unaffected controls','population']
diagss = [['diag_PDHC','diag_ProdHC','diag_PDProdHC'],
          ['diag_PDallHC','diag_ProdallHC','diag_PDProdallHC'],
          ['diag_PDPopulationNoProd','diag_ProdPopulationNoPD','diag_PDProdPopulation']]
labels = ['Diagnosed','Prodromal','Diagnosed & Prodromal']
subfigs = ['A','B','C']

for save,control,diags in zip(saves,controls,diagss):
    fig,axes = plt.subplots(nrows=3,ncols=1,figsize=(15,25))
    for diag,ax,label,subfig in zip(diags,axes.flatten(),labels,subfigs):
        print(diag,control)
        plot_AUROC_CV(curves.loc[(diag,slice(None)),:].droplevel(level=0,axis=0),pf.loc[(diag,slice(None)),:].droplevel(level=0,axis=0),
                      features=['intercept','genetics+family','lifestyle_nofam','blood','prodromalsigns_beforePD','all_acc_features',
                                 'all_acc_features+blood+lifestyle+genetics+prodromalsigns_beforePD'],
                      fnames=['no-skill','genetics+family','lifestyle','blood','prodromal symptoms',
                                                'all accelerometer','combined'],ax=ax,label=label,control=control)
        ax.text(-0.1, 1, subfig, transform=ax.transAxes, 
                size=20, weight='bold')

    if save:
        plt.savefig(f'{image_path}/{save}.png',dpi=300,bbox_inches='tight')
        plt.savefig(f'{image_path}/{save}.pdf',dpi=300,bbox_inches='tight')